torch → PyTorch, la librería principal para deep learning.

DataLoader → Permite cargar datos en batches para entrenamiento/evaluación.

random_split → Permite dividir un dataset en subsets (train/val/test).

GestosDataset → Clase personalizada que hereda de torch.utils.data.Dataset para manejar imágenes de gestos.

transformacion → Transformaciones/augmentations centralizadas para el dataset (normalización, aumentos, /etc.).

In [7]:
import torch
import os
from torch.utils.data import DataLoader, random_split
from dataset_utils import GestosDataset, transformacion, transformacion_contornos

Verificamos si PyTorch puede usar GPU (CUDA).

In [8]:
# Verificar GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)
if device.type == "cuda":
    print("GPU disponible:", torch.cuda.get_device_name(0))
    print("Versión CUDA:", torch.version.cuda)

Usando dispositivo: cuda
GPU disponible: NVIDIA GeForce GTX 1060 6GB
Versión CUDA: 11.8


In [9]:
# Ruta a la carpeta con todas las clases
carpeta1 = r"E:\Mis Documentos\Christian\Tesis\tesis_gestos\data\Clases_200"

# Mas imagenes para reforzar el dataset 
carpeta2 = r"E:\Mis Documentos\Christian\Tesis\tesis_gestos\data\Clases_200_comp"

carpeta_modelos = "modelos"
os.makedirs(carpeta_modelos, exist_ok=True)

Clases_200/

 ├── Sub-carpetas para las clases

Se utiliza la clase `GestosDataset` para cargar todas las imágenes del dataset, aplicando las transformaciones.  
Luego se divide en entrenamiento (80%) y validación (20%).  

In [10]:
# ===============================
# Crear Dataset y DataLoaders
# ===============================

# Crear dataset completo
dataset_gestos = GestosDataset([carpeta1, carpeta2], transformacion = transformacion)

# Dividir dataset en train y validación (80% - 20%)
train_size = int(0.8 * len(dataset_gestos))
val_size   = len(dataset_gestos) - train_size
train_dataset, val_dataset = random_split(dataset_gestos, [train_size, val_size])

# DataLoaders para iterar en batchs
train_loader = DataLoader(train_dataset, 
                          batch_size = 32, 
                          shuffle = True, 
                          num_workers = 2)

val_loader = DataLoader(val_dataset, 
                        batch_size = 32, 
                        shuffle = False, 
                        num_workers = 2)

# Tamaños para verificar
print("Tamaño dataset completo:", len(dataset_gestos))
print("Tamaño train:", len(train_dataset))
print("Tamaño val:", len(val_dataset))

torch.save(dataset_gestos, os.path.join(carpeta_modelos, "dataset_gestos.pt"))

Tamaño dataset completo: 12911
Tamaño train: 10328
Tamaño val: 2583


**Notas**
__________________________________________________________________________________________________
DataLoader --> Forma estándar de iterar sobre un dataset en batches durante el entrenamiento y validación.
 - Batch_size: Divide el dataset en lotes (32).
 - Shuffle: Elegir si el modelo aprende patrones por el orden de las imágenes.
 - num_workers: Mientras el modelo entrena con un batch, el DataLoader ya está preparando el siguiente, acortando tiempos.

__________________________________________________________________________________________________
*torch.save(dataset_gestos, "dataset_gestos.pt")*, guarda el dataset preprocesado para reutilizarlo sin volver a cargar imágenes desde cero.

*Archivo*: dataset_gestos.pt